# 第4章: 言語解析

問題30から問題35までは、以下の文章`text`（太宰治の『走れメロス』の冒頭部分）に対して、言語解析を実施せよ。問題36から問題39までは、国家を説明した文書群（日本語版ウィキペディア記事から抽出したテキスト群）をコーパスとして、言語解析を実施せよ。

In [1]:
text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""

In [2]:
!pip install janome

## 30. 動詞
文章`text`に含まれる動詞をすべて表示せよ。

In [3]:
from janome.tokenizer import Tokenizer
t = Tokenizer()
for token in t.tokenize(text):
  if token.part_of_speech.split(',')[0] == '動詞':
    print(token.surface)

し
除か
なら
し
わから
吹き
遊ん
暮し
来


## 31. 動詞の原型
文章`text`に含まれる動詞と、その原型をすべて表示せよ。

In [45]:
for token in t.tokenize(text):
  if token.part_of_speech.split(',')[0] == '動詞':
    print("動詞: ", token.surface, "\t", "原型: " ,token.base_form)

動詞:  し 	 原型:  する
動詞:  除か 	 原型:  除く
動詞:  なら 	 原型:  なる
動詞:  し 	 原型:  する
動詞:  わから 	 原型:  わかる
動詞:  吹き 	 原型:  吹く
動詞:  遊ん 	 原型:  遊ぶ
動詞:  暮し 	 原型:  暮す
動詞:  来 	 原型:  来る


## 32. 「AのB」
文章`text`において、2つの名詞が「の」で連結されている名詞句をすべて抽出せよ。

In [49]:
takens = list(t.tokenize(text))
for i in range(len(takens)-2):
  t1 = takens[i]
  t2 = takens[i+1]
  t3 = takens[i+2]
  if (
      t1.part_of_speech.split(',')[0] == '名詞' and
      t2.surface == 'の' and
      t3.part_of_speech.split(',')[0] == '名詞'
  ):
    print(t1.surface + t2.surface + t3.surface)

暴虐の王
村の牧人


## 33. 係り受け解析

文章`text`に係り受け解析を適用し、係り元と係り先のトークン（形態素や文節などの単位）をタブ区切り形式ですべて抽出せよ。

In [4]:
!pip install -U spacy==3.7.2
!pip install -U ginza ja_ginza

In [6]:
import spacy
import ginza
import ja_ginza

nlp = spacy.load("ja_ginza")

doc = nlp(text)

for sent in doc.sents:
    for token in sent:
        # token が係り元、token.head が係り先
        if token != token.head:
            print(token.text, "\t", token.head.text)


 	 メロス
メロス 	 激怒
は 	 メロス
し 	 激怒
た 	 激怒
。 	 激怒
必ず 	 除か
、 	 必ず
かの 	 暴虐
邪智 	 暴虐
暴虐 	 王
の 	 暴虐
王 	 除か
を 	 王
除か 	 決意
なけれ 	 除か
ば 	 なけれ
なら 	 なけれ
ぬ 	 なけれ
と 	 除か
し 	 決意
た 	 決意
。 	 決意

 	 メロス
メロス 	 わから
に 	 メロス
は 	 メロス
政治 	 わから
が 	 政治
ぬ 	 わから
。 	 わから

 	 メロス
メロス 	 牧人
は 	 メロス
、 	 メロス
村 	 牧人
の 	 村
で 	 牧人
ある 	 で
。 	 牧人

 	 笛
笛 	 吹き
を 	 笛
吹き 	 暮し
、 	 吹き
羊 	 遊ん
と 	 羊
遊ん 	 暮し
で 	 遊ん
て 	 暮し
来 	 て
た 	 暮し
。 	 暮し

 	 邪悪
けれど 	 

も 	 

邪悪 	 敏感
に 	 邪悪
対し 	 に
ては 	 に
、 	 邪悪
人 	 倍
一 	 倍
倍 	 敏感
に 	 倍
で 	 敏感
あっ 	 で
た 	 敏感
。 	 敏感


## 34. 主述の関係
文章`text`において、「メロス」が主語であるときの述語を抽出せよ。

In [7]:
for sent in doc.sents:
    for token in sent:
        # 「メロス」が主語のとき
        if token.text == "メロス" and token.dep_ == "nsubj":
            predicate = token.head
            print(predicate.text)

激怒
牧人


## 35. 係り受け木
「メロスは激怒した。」の係り受け木を可視化せよ。

In [8]:
import spacy
from spacy import displacy

nlp = spacy.load("ja_ginza")

doc = nlp("メロスは激怒した。")

displacy.render(doc, style="dep", jupyter=True)

## 36. 単語の出現頻度

問題36から39までは、Wikipediaの記事を以下のフォーマットで書き出したファイル[jawiki-country.json.gz](/data/jawiki-country.json.gz)をコーパスと見なし、統計的な分析を行う。

* 1行に1記事の情報がJSON形式で格納される
* 各行には記事名が"title"キーに、記事本文が"text"キーの辞書オブジェクトに格納され、そのオブジェクトがJSON形式で書き出される
* ファイル全体はgzipで圧縮される

まず、第3章の処理内容を参考に、Wikipedia記事からマークアップを除去し、各記事のテキストを抽出せよ。そして、コーパスにおける単語（形態素）の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 37. 名詞の出現頻度
コーパスにおける名詞の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 38. TF・IDF
日本に関する記事における名詞のTF・IDFスコアを求め、TF・IDFスコア上位20語とそのTF, IDF, TF・IDFを表示せよ。

## 39. Zipfの法則
コーパスにおける単語の出現頻度順位を横軸、その出現頻度を縦軸として、両対数グラフをプロットせよ。